# Set Piece Analytics — ETL

Este notebook extrai eventos de bolas paradas do StatsBomb Open Data.

**Saída:**
- `data_processed/<competicao>/<temporada>.parquet` — um arquivo por competição/temporada
- `data_processed/set_pieces.parquet` — arquivo centralizado com tudo concatenado

**Execute as células em ordem.** As células 1–4 são de exploração e não gravam nada. O processamento começa na célula 5.

---

## Célula 1 — Imports e configurações

In [5]:
import os
import re
import pandas as pd
from statsbombpy import sb
from tqdm import tqdm

OUTPUT_DIR = "data_processed"
CENTRAL_FILE = os.path.join(OUTPUT_DIR, "set_pieces.parquet")

os.makedirs(OUTPUT_DIR, exist_ok=True)


def slugify(text):
    """Converte nome de competição/temporada para nome de pasta/arquivo seguro."""
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


print("Ambiente pronto.")
print(f"Exemplos de slugify: 'La Liga' → '{slugify('La Liga')}', '2017/2018' → '{slugify('2017/2018')}'")

Ambiente pronto.
Exemplos de slugify: 'La Liga' → 'la_liga', '2017/2018' → '2017_2018'


## Célula 2 — Explorar competições disponíveis

In [6]:
competitions = sb.competitions()
print(f"Total de competições/temporadas: {len(competitions)}")
competitions[["competition_id", "competition_name", "season_id", "season_name"]].head(5)

c:\Users\I764251\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Total de competições/temporadas: 80


,competition_id,competition_name,season_id,season_name
0,9,1. Bundesliga,281,2023/2024
1,9,1. Bundesliga,27,2015/2016
2,1267,African Cup of Nations,107,2023
3,16,Champions League,4,2018/2019
4,16,Champions League,1,2017/2018


## Célula 3 — Teste com uma única partida

Antes de processar tudo, validamos a estrutura dos dados e as colunas disponíveis.

In [7]:
# La Liga 2017/18: competition_id=11, season_id=1
sample_matches = sb.matches(competition_id=11, season_id=1)
sample_match_id = int(sample_matches["match_id"].iloc[0])
print(f"Partida de teste: {sample_match_id}")

sample_events = sb.events(match_id=sample_match_id)
print(f"Total de eventos: {len(sample_events)}")
print(f"Colunas disponíveis:")
print(sample_events.columns.tolist())

c:\Users\I764251\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Partida de teste: 9880


c:\Users\I764251\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Total de eventos: 3947
Colunas disponíveis:
['50_50', 'ball_receipt_outcome', 'ball_recovery_recovery_failure', 'block_deflection', 'block_save_block', 'carry_end_location', 'clearance_aerial_won', 'clearance_body_part', 'clearance_head', 'clearance_left_foot', 'clearance_right_foot', 'counterpress', 'dribble_nutmeg', 'dribble_outcome', 'duel_outcome', 'duel_type', 'duration', 'foul_committed_advantage', 'foul_committed_card', 'foul_committed_penalty', 'foul_won_advantage', 'foul_won_defensive', 'foul_won_penalty', 'goalkeeper_body_part', 'goalkeeper_end_location', 'goalkeeper_outcome', 'goalkeeper_position', 'goalkeeper_shot_saved_to_post', 'goalkeeper_technique', 'goalkeeper_type', 'id', 'index', 'interception_outcome', 'location', 'match_id', 'minute', 'off_camera', 'out', 'pass_aerial_won', 'pass_angle', 'pass_assisted_shot_id', 'pass_body_part', 'pass_cross', 'pass_cut_back', 'pass_deflected', 'pass_end_location', 'pass_goal_assist', 'pass_height', 'pass_inswinging', 'pass_length'

In [8]:
# Verificar set pieces presentes na partida de teste
passes = sample_events[sample_events["type"] == "Pass"]
shots = sample_events[sample_events["type"] == "Shot"]

print("=== Tipos de Pass ===")
print(passes["pass_type"].value_counts())

print("\n=== Tipos de Shot ===")
print(shots["shot_type"].value_counts())

print("\n=== Colunas de shot ===")
shot_cols = [c for c in sample_events.columns if c.startswith("shot")]
print(shot_cols)

=== Tipos de Pass ===
pass_type
Recovery        55
Throw-in        35
Goal Kick       22
Free Kick       19
Corner          11
Kick Off         5
Interception     5
Name: count, dtype: int64

=== Tipos de Shot ===
shot_type
Open Play    27
Free Kick     1
Penalty       1
Name: count, dtype: int64

=== Colunas de shot ===
['shot_aerial_won', 'shot_body_part', 'shot_deflected', 'shot_end_location', 'shot_first_time', 'shot_freeze_frame', 'shot_key_pass_id', 'shot_outcome', 'shot_saved_to_post', 'shot_statsbomb_xg', 'shot_technique', 'shot_type']


## Célula 4 — Funções de extração

Cada função recebe o DataFrame de eventos de uma partida e retorna uma lista de dicionários, um por oportunidade de bola parada.

In [9]:
MINUTE_BINS = [0, 15, 30, 45, 60, 75, 91]
MINUTE_LABELS = ["0-15", "16-30", "31-45", "46-60", "61-75", "76-90+"]


def get_minute_band(minute):
    for i, upper in enumerate(MINUTE_BINS[1:]):
        if minute <= upper:
            return MINUTE_LABELS[i]
    return "76-90+"


def safe_loc(location):
    if isinstance(location, list) and len(location) >= 2:
        return location[0], location[1]
    return None, None


def extract_pass_based(events_df, pass_type_value, set_piece_label, match_meta):
    records = []
    trigger_passes = events_df[
        (events_df["type"] == "Pass") &
        (events_df["pass_type"] == pass_type_value)
    ]

    for _, trigger in trigger_passes.iterrows():
        possession_num = trigger["possession"]
        possession_events = events_df[events_df["possession"] == possession_num]
        shots_in_possession = possession_events[possession_events["type"] == "Shot"]

        shots_generated = 1 if len(shots_in_possession) > 0 else 0
        goals_scored = 1 if (shots_in_possession["shot_outcome"] == "Goal").any() else 0
        xg_sum = shots_in_possession["shot_statsbomb_xg"].sum() if shots_generated else 0.0

        shot_x, shot_y = None, None
        if shots_generated:
            shot_x, shot_y = safe_loc(shots_in_possession.iloc[0].get("location"))

        origin_x, origin_y = safe_loc(trigger.get("location"))

        records.append({
            "match_id": match_meta["match_id"],
            "competition_name": match_meta["competition_name"],
            "season_name": match_meta["season_name"],
            "team_name": trigger["team"],
            "set_piece_type": set_piece_label,
            "period": trigger["period"],
            "minute": trigger["minute"],
            "minute_band": get_minute_band(trigger["minute"]),
            "shots_generated": shots_generated,
            "goals_scored": goals_scored,
            "xg_sum": float(xg_sum),
            "origin_x": origin_x,
            "origin_y": origin_y,
            "shot_x": shot_x,
            "shot_y": shot_y,
        })

    return records


def extract_shot_based(events_df, shot_type_value, set_piece_label, match_meta):
    records = []
    trigger_shots = events_df[
        (events_df["type"] == "Shot") &
        (events_df["shot_type"] == shot_type_value)
    ]

    for _, shot in trigger_shots.iterrows():
        goals_scored = 1 if shot.get("shot_outcome") == "Goal" else 0
        xg = shot.get("shot_statsbomb_xg", 0.0)
        shot_x, shot_y = safe_loc(shot.get("location"))

        records.append({
            "match_id": match_meta["match_id"],
            "competition_name": match_meta["competition_name"],
            "season_name": match_meta["season_name"],
            "team_name": shot["team"],
            "set_piece_type": set_piece_label,
            "period": shot["period"],
            "minute": shot["minute"],
            "minute_band": get_minute_band(shot["minute"]),
            "shots_generated": 1,
            "goals_scored": goals_scored,
            "xg_sum": float(xg) if pd.notna(xg) else 0.0,
            "origin_x": shot_x,
            "origin_y": shot_y,
            "shot_x": shot_x,
            "shot_y": shot_y,
        })

    return records


def extract_set_pieces(events_df, match_meta):
    records = []
    records += extract_pass_based(events_df, "Corner", "Escanteio", match_meta)
    records += extract_pass_based(events_df, "Free Kick", "Falta Indireta", match_meta)
    records += extract_pass_based(events_df, "Throw-in", "Lateral", match_meta)
    records += extract_shot_based(events_df, "Free Kick", "Falta Direta", match_meta)
    records += extract_shot_based(events_df, "Penalty", "Penalti", match_meta)
    return records


# --- Teste das funções com a partida de amostra ---
sample_match_row = sample_matches.iloc[0]
sample_meta = {
    "match_id": int(sample_match_row["match_id"]),
    "competition_name": sample_match_row["competition"],
    "season_name": sample_match_row["season"],
}

test_records = extract_set_pieces(sample_events, sample_meta)
test_df = pd.DataFrame(test_records)

print(f"Set pieces extraídos da partida de teste: {len(test_df)}")
print(test_df["set_piece_type"].value_counts())
test_df.head()

Set pieces extraídos da partida de teste: 67
set_piece_type
Lateral           35
Falta Indireta    19
Escanteio         11
Falta Direta       1
Penalti            1
Name: count, dtype: int64


,match_id,competition_name,season_name,team_name,set_piece_type,period,minute,minute_band,shots_generated,goals_scored,xg_sum,origin_x,origin_y,shot_x,shot_y
0,9880,Spain - La Liga,2017/2018,Barcelona,Escanteio,1,9,0-15,0,0,0.000000,120.0,80.0,NaN,NaN
1,9880,Spain - La Liga,2017/2018,Barcelona,Escanteio,1,16,16-30,0,0,0.000000,120.0,0.1,NaN,NaN
2,9880,Spain - La Liga,2017/2018,Valencia,Escanteio,1,20,16-30,0,0,0.000000,120.0,0.1,NaN,NaN
3,9880,Spain - La Liga,2017/2018,Valencia,Escanteio,1,22,16-30,1,0,0.017197,120.0,80.0,92.0,39.7
4,9880,Spain - La Liga,2017/2018,Valencia,Escanteio,1,23,16-30,0,0,0.000000,120.0,80.0,NaN,NaN


## Célula 5 — Loop completo

Processa todas as competições e temporadas. Cada combinação é salva em:
`data_processed/<competicao>/<temporada>.parquet`

Se o loop for interrompido, os arquivos já gerados são preservados e podem ser reaproveitados na célula 7.

> **Atenção:** requer conexão com internet. Para ~4.235 partidas pode levar vários minutos.

In [10]:
errors = []
skipped = []

for _, comp_row in tqdm(competitions.iterrows(), total=len(competitions), desc="Competições"):
    comp_id = int(comp_row["competition_id"])
    season_id = int(comp_row["season_id"])
    comp_name = comp_row["competition_name"]
    season_name = comp_row["season_name"]

    comp_slug = slugify(comp_name)
    season_slug = slugify(season_name)
    comp_dir = os.path.join(OUTPUT_DIR, comp_slug)
    season_file = os.path.join(comp_dir, f"{season_slug}.parquet")

    # Pular se já foi processado (permite retomar de onde parou)
    if os.path.exists(season_file):
        skipped.append(f"{comp_name} / {season_name}")
        continue

    os.makedirs(comp_dir, exist_ok=True)

    try:
        matches_df = sb.matches(competition_id=comp_id, season_id=season_id)
    except Exception as e:
        errors.append({"where": f"matches({comp_name} / {season_name})", "error": str(e)})
        continue

    season_records = []
    for _, match_row in matches_df.iterrows():
        match_id = int(match_row["match_id"])
        match_meta = {
            "match_id": match_id,
            "competition_name": comp_name,
            "season_name": season_name,
        }
        try:
            events_df = sb.events(match_id=match_id)
            season_records.extend(extract_set_pieces(events_df, match_meta))
        except Exception as e:
            errors.append({"where": f"events(match_id={match_id})", "error": str(e)})

    if season_records:
        pd.DataFrame(season_records).to_parquet(season_file, index=False)

print(f"\nConcluído.")
if skipped:
    print(f"Já existiam e foram pulados: {len(skipped)}")
if errors:
    print(f"Erros encontrados: {len(errors)}")
    for err in errors[:5]:
        print(f"  - {err}")

Competições:   0%|          | 0/80 [00:00<?, ?it/s]c:\Users\I764251\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\I764251\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\I764251\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\I764251\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(
c:\Users\I764251\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(

KeyboardInterrupt: 

## Célula 6 — Validação

Lê todos os parquets individuais e valida antes de gerar o arquivo centralizado.

In [11]:
# Encontrar todos os parquets individuais gerados
individual_files = []
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in files:
        path = os.path.join(root, f)
        # Excluir o centralizado (se já existir)
        if f.endswith(".parquet") and path != CENTRAL_FILE:
            individual_files.append(path)

print(f"Arquivos individuais encontrados: {len(individual_files)}")

# Carregar e concatenar para validação
df = pd.concat([pd.read_parquet(f) for f in individual_files], ignore_index=True)

print(f"\n=== Shape total ===")
print(df.shape)

print("\n=== Set pieces por tipo ===")
print(df["set_piece_type"].value_counts())

print("\n=== Competições presentes ===")
print(df["competition_name"].value_counts())

print("\n=== Nulos por coluna ===")
print(df.isnull().sum())

print("\n=== xG quando há shot (deve ser > 0 na maioria) ===")
has_shot = df[df["shots_generated"] == 1]
print(f"Linhas com shot: {len(has_shot)}")
print(f"xg_sum = 0 quando há shot: {(has_shot['xg_sum'] == 0).sum()}")

df.head()

Arquivos individuais encontrados: 26

=== Shape total ===
(24758, 15)

=== Set pieces por tipo ===
set_piece_type
Lateral           13394
Falta Indireta     7897
Escanteio          2971
Falta Direta        290
Penalti             206
Name: count, dtype: int64

=== Competições presentes ===
competition_name
FA Women's Super League    10380
1. Bundesliga               5493
African Cup of Nations      4369
Copa America                2505
Champions League            1723
Copa del Rey                 288
Name: count, dtype: int64

=== Nulos por coluna ===
match_id                0
competition_name        0
season_name             0
team_name               0
set_piece_type          0
period                  0
minute                  0
minute_band             0
shots_generated         0
goals_scored            0
xg_sum                  0
origin_x                0
origin_y                0
shot_x              20727
shot_y              20727
dtype: int64

=== xG quando há shot (deve ser > 0 na

,match_id,competition_name,season_name,team_name,set_piece_type,period,minute,minute_band,shots_generated,goals_scored,xg_sum,origin_x,origin_y,shot_x,shot_y
0,3890563,1. Bundesliga,2015/2016,Ingolstadt,Escanteio,1,9,0-15,0,0,0.000000,120.0,80.0,NaN,NaN
1,3890563,1. Bundesliga,2015/2016,Ingolstadt,Escanteio,1,35,31-45,0,0,0.000000,120.0,80.0,NaN,NaN
2,3890563,1. Bundesliga,2015/2016,Ingolstadt,Escanteio,2,46,46-60,0,0,0.000000,120.0,80.0,NaN,NaN
3,3890563,1. Bundesliga,2015/2016,Bayer Leverkusen,Escanteio,2,51,46-60,0,0,0.000000,120.0,80.0,NaN,NaN
4,3890563,1. Bundesliga,2015/2016,Bayer Leverkusen,Escanteio,2,81,76-90+,1,0,0.196415,120.0,80.0,109.3,44.2


## Célula 7 — Gerar arquivo centralizado

Concatena todos os parquets individuais em `data_processed/set_pieces.parquet`.

In [12]:
df.to_parquet(CENTRAL_FILE, index=False)
file_size_mb = os.path.getsize(CENTRAL_FILE) / (1024 * 1024)
print(f"Salvo em: {CENTRAL_FILE}")
print(f"Tamanho: {file_size_mb:.2f} MB")
print(f"Linhas: {len(df):,}")

Salvo em: data_processed\set_pieces.parquet
Tamanho: 0.19 MB
Linhas: 24,758


## Célula 8 — Verificações finais

Confirma que o arquivo centralizado pode ser lido e calcula os KPIs globais.

In [13]:
df_check = pd.read_parquet(CENTRAL_FILE)

print("=== Leitura do parquet centralizado OK ===")
print(f"Shape: {df_check.shape}")
print(f"Colunas: {df_check.columns.tolist()}")

print("\n=== KPIs globais ===")
total = len(df_check)
tf = df_check["shots_generated"].sum() / total
tc = df_check["goals_scored"].sum() / total
xg_medio = df_check["xg_sum"].sum() / total

print(f"Total de bolas paradas: {total:,}")
print(f"Taxa de Finalização (TF): {tf:.2%}")
print(f"Taxa de Conversão (TC): {tc:.2%}")
print(f"xG Médio: {xg_medio:.4f}")

print("\n=== Estrutura gerada em data_processed/ ===")
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, "").count(os.sep)
    indent = "  " * level
    folder = os.path.basename(root)
    print(f"{indent}{folder}/")
    for f in sorted(files):
        size_kb = os.path.getsize(os.path.join(root, f)) / 1024
        print(f"{indent}  {f}  ({size_kb:.0f} KB)")

=== Leitura do parquet centralizado OK ===
Shape: (24758, 15)
Colunas: ['match_id', 'competition_name', 'season_name', 'team_name', 'set_piece_type', 'period', 'minute', 'minute_band', 'shots_generated', 'goals_scored', 'xg_sum', 'origin_x', 'origin_y', 'shot_x', 'shot_y']

=== KPIs globais ===
Total de bolas paradas: 24,758
Taxa de Finalização (TF): 16.28%
Taxa de Conversão (TC): 2.24%
xG Médio: 0.0219

=== Estrutura gerada em data_processed/ ===
data_processed/
  set_pieces.parquet  (197 KB)
  1_bundesliga/
    2015_2016.parquet  (39 KB)
    2023_2024.parquet  (33 KB)
  african_cup_of_nations/
    2023.parquet  (50 KB)
  champions_league/
    1970_1971.parquet  (10 KB)
    1971_1972.parquet  (11 KB)
    1972_1973.parquet  (11 KB)
    1999_2000.parquet  (10 KB)
    2003_2004.parquet  (11 KB)
    2004_2005.parquet  (11 KB)
    2006_2007.parquet  (11 KB)
    2008_2009.parquet  (11 KB)
    2009_2010.parquet  (10 KB)
    2010_2011.parquet  (10 KB)
    2011_2012.parquet  (11 KB)
    2012_2